# Day 45: Phase 3 Capstone: Doubly Robust (AIPW) Estimation

*Beginner to Ninja: Math for Causal Inference*

Combining an outcome model with a propensity model so that the estimate is right if either one is. The bridge from Phase 3 to double machine learning.

**Learning outcomes**

- Write the regression (outcome-model), IPW and augmented IPW (AIPW) estimators of the ATE.
- Prove that AIPW is consistent if either the outcome model or the propensity model is correct.
- Compute AIPW estimates and influence-function standard errors by hand.
- Choose among Phase 3 designs and estimators for a given policy question.

## Theory

### Two ways to use unconfoundedness

With $\mu_t(x) = E[Y \mid T = t, X = x]$ and $e(x) = P(T = 1 \mid X = x)$:

- **Outcome regression (imputation):** $\hat\tau_{REG} = \frac1n\sum_i\big(\hat\mu_1(X_i) - \hat\mu_0(X_i)\big)$. Right if the outcome models are right.
- **IPW (Day 44):** right if the propensity model is right.

### Augmented IPW

Start from the regression estimate and add an IPW-weighted average of the outcome-model errors:

$$
\hat\tau_{AIPW} = \frac1n\sum_i\Big[\hat\mu_1(X_i) - \hat\mu_0(X_i) + \frac{T_i(Y_i - \hat\mu_1(X_i))}{\hat e(X_i)} - \frac{(1 - T_i)(Y_i - \hat\mu_0(X_i))}{1 - \hat e(X_i)}\Big].
$$

The correction terms repair whatever bias the outcome model leaves.

### Double robustness

$\hat\tau_{AIPW}$ is consistent if **either**

- the outcome models $\mu_0, \mu_1$ are correct (whatever $\hat e$ is), **or**
- the propensity model $e$ is correct (whatever $\hat\mu$ is).

When both are correct, it is also **efficient**: it attains the smallest possible asymptotic variance for the ATE under unconfoundedness (the semiparametric bound, Day 88).

### Inference

Write $\hat\psi_i$ for the bracketed term of unit $i$, so $\hat\tau_{AIPW} = \frac1n\sum_i\hat\psi_i$. Its standard error is approximately

$$
\operatorname{SE} \approx \frac{\operatorname{sd}(\hat\psi_i)}{\sqrt n}.
$$

The $\hat\psi_i$ are (estimated) **influence-function** values (Day 87). With flexible machine-learning models for $\mu$ and $e$, fit them on different folds from those used to evaluate $\hat\psi_i$ (**cross-fitting**): this is double/debiased machine learning (Day 90).

### Phase 3 in one table

| Situation | Tool (Day) |
|---|---|
| You control assignment | complete or blocked randomisation (35, 38), Neyman variance (36) |
| Tiny sample or sharp hypothesis | Fisher randomisation test (37) |
| Baseline covariates in an RCT | Lin adjustment (39) |
| Treatment delivered to groups or spillovers | cluster randomisation (40), saturation designs (32) |
| People ignore their assignment | ITT and complier effects (41) |
| Observational data, rich covariates | unconfoundedness + overlap (42), propensity score (43), IPW (44), AIPW (45) |

## Part 1 · Warm-up

### Exercise 1 · One treated unit's contribution

A treated household has $Y = 20$, $\hat\mu_1 = 19$, $\hat\mu_0 = 15$, $\hat e = 0.8$. Compute its AIPW term $\hat\psi_i$.

<details><summary>Show solution</summary>

**Step 1.** Regression part: $\hat\mu_1 - \hat\mu_0 = 4$.

**Step 2.** Correction (treated): $(Y - \hat\mu_1)/\hat e = (20 - 19)/0.8 = 1.25$.

**Step 3.** $\hat\psi_i = 4 + 1.25 = 5.25$.

</details>

*Your work:*

### Exercise 2 · One control unit's contribution

A control household has $Y = 14$, $\hat\mu_1 = 19$, $\hat\mu_0 = 15$, $\hat e = 0.8$. Compute $\hat\psi_i$.

<details><summary>Show solution</summary>

**Step 1.** Regression part: $19 - 15 = 4$.

**Step 2.** Correction (control): $-(Y - \hat\mu_0)/(1 - \hat e) = -(14 - 15)/0.2 = +5$.

**Step 3.** $\hat\psi_i = 4 + 5 = 9$.

**Answer:** 9. A rare control (only 20% of similar households are untreated) with an outcome below its prediction pulls the estimate up strongly.

</details>

*Your work:*

### Exercise 3 · Reading the correction

What does the correction term do if the outcome model predicts the treated outcomes perfectly?

<details><summary>Show solution</summary>

**Step 1.** Then $Y_i - \hat\mu_1(X_i) = 0$ for every treated unit (and similarly for controls).

**Answer:** the corrections vanish and AIPW equals the regression estimator; weights matter only to the extent the outcome model errs.

</details>

*Your work:*

### Exercise 4 · A standard error

An AIPW analysis of 900 workers gives $\hat\psi_i$ values with sd 12 (\$1,000). Compute the SE.

<details><summary>Show solution</summary>

**Step 1.** SE $\approx 12/\sqrt{900} = 12/30 = 0.4$.

**Answer:** \$400.

</details>

*Your work:*

### Exercise 5 · Which model matters?

You are confident about how a training programme selected participants (eligibility rules recorded in the files) but not about how earnings depend on covariates. Which estimator is safest?

<details><summary>Show solution</summary>

**Step 1.** Knowledge of selection means the propensity model is likely right; the outcome model may be wrong.

**Step 2.** IPW and AIPW both rely on a correct propensity model; AIPW also uses the outcome model to reduce variance.

**Answer:** AIPW (or IPW): both stay consistent through the propensity model, and AIPW is more precise whenever the outcome model explains a good share of earnings variation.

</details>

## Part 2 · Proofs

*Your work:*

### Exercise 6 · Robust to a wrong propensity model

Assume unconfoundedness and that $\tilde\mu_1(x) = \mu_1(x)$ is correct, while $\tilde e(x)$ is any function in $(0, 1)$. Prove $E\big[\tilde\mu_1(X) + T(Y - \tilde\mu_1(X))/\tilde e(X)\big] = E[Y(1)]$.

<details><summary>Show solution</summary>

**Step 1.** $E[\mu_1(X)] = E[E[Y \mid T = 1, X]] = E[Y(1)]$ (Day 42).

**Step 2.** Correction term, by the LIE: $E\Big[\frac{T(Y - \mu_1(X))}{\tilde e(X)}\Big] = E\Big[\frac{E[T(Y - \mu_1(X)) \mid X]}{\tilde e(X)}\Big]$.

**Step 3.** $E[T(Y - \mu_1(X)) \mid X] = e(X)\,\big(E[Y \mid T = 1, X] - \mu_1(X)\big) = 0$.

**Step 4.** So the correction has mean 0 whatever $\tilde e$ is, and the total is $E[Y(1)]$. $\blacksquare$

</details>

*Your work:*

### Exercise 7 · Robust to a wrong outcome model

Now assume $\tilde e(x) = e(x)$ is correct and $\tilde\mu_1$ is any function. Prove the same identity.

<details><summary>Show solution</summary>

**Step 1.** Rearrange: $\tilde\mu_1(X) + \frac{T(Y - \tilde\mu_1(X))}{e(X)} = \frac{TY}{e(X)} + \tilde\mu_1(X)\Big(1 - \frac{T}{e(X)}\Big)$.

**Step 2.** $E[TY/e(X)] = E[Y(1)]$ (Day 44, Exercise 6).

**Step 3.** $E\big[\tilde\mu_1(X)(1 - T/e(X)) \mid X\big] = \tilde\mu_1(X)\big(1 - e(X)/e(X)\big) = 0$.

**Step 4.** So the total has mean $E[Y(1)]$. $\blacksquare$ The same arguments for $Y(0)$ give double robustness for the ATE.

</details>

*Your work:*

### Exercise 8 · AIPW nests IPW

Show that AIPW with $\hat\mu_0 = \hat\mu_1 = 0$ is the Horvitz–Thompson IPW estimator.

<details><summary>Show solution</summary>

**Step 1.** With $\hat\mu \equiv 0$, the regression part is 0.

**Step 2.** The corrections become $T_iY_i/\hat e_i - (1 - T_i)Y_i/(1 - \hat e_i)$.

**Step 3.** Their average is $\hat\tau_{HT}$ (Day 44). $\blacksquare$

A good outcome model shrinks the residuals that get weighted, which is why AIPW is typically less variable than IPW when the outcome model fits well.

</details>

*Your work:*

### Exercise 9 · AIPW in a randomised experiment

In an RCT with known $e = p$ and linear outcome models fitted separately by arm, explain why AIPW is consistent whatever the outcome model, and how it relates to Lin's estimator (Day 39).

<details><summary>Show solution</summary>

**Step 1.** The propensity is known exactly ($e = p$), so Exercise 7 applies: consistent for any $\hat\mu$.

**Step 2.** With linear $\hat\mu_t$ fitted by OLS in each arm, residuals average to zero within each arm, so the correction terms average to (nearly) zero.

**Step 3.** AIPW then reduces to $\frac1n\sum(\hat\mu_1(X_i) - \hat\mu_0(X_i))$: each arm's regression line evaluated at every unit's covariates and averaged, which equals Lin's estimator.

**Answer:** in an RCT, covariate adjustment is a special case of AIPW: it can only improve precision, never introduce bias asymptotically.

</details>

*Your work:*

### Exercise 10 · The product of errors

Show that, for the $Y(1)$ part, the bias of AIPW with wrong models $\tilde\mu_1$ and $\tilde e$ is $E\Big[\frac{(e(X) - \tilde e(X))(\mu_1(X) - \tilde\mu_1(X))}{\tilde e(X)}\Big]$.

<details><summary>Show solution</summary>

**Step 1.** Condition on $X$: $E[T(Y - \tilde\mu_1) \mid X] = e(X)(\mu_1(X) - \tilde\mu_1(X))$.

**Step 2.** So $E\big[\tilde\mu_1 + T(Y - \tilde\mu_1)/\tilde e\big] = E\Big[\tilde\mu_1 + \frac{e}{\tilde e}(\mu_1 - \tilde\mu_1)\Big]$.

**Step 3.** Subtract $E[\mu_1(X)] = E[Y(1)]$: bias $= E\Big[(\mu_1 - \tilde\mu_1)\Big(\frac{e}{\tilde e} - 1\Big)\Big] = E\Big[\frac{(e - \tilde e)(\mu_1 - \tilde\mu_1)}{\tilde e}\Big]$. $\blacksquare$

The bias is a *product* of the two errors: if each model is moderately good, the bias is very small. This is why AIPW works with machine-learning nuisance models (Day 90).

</details>

## Part 3 · Applied

Return to Day 44's six households, now with outcome-model predictions:

| Household | $T$ | $\hat e$ | $Y$ | $\hat\mu_1$ | $\hat\mu_0$ |
|---|---|---|---|---|---|
| 1 | 1 | 0.8 | 20 | 19 | 15 |
| 2 | 1 | 0.5 | 16 | 16 | 12 |
| 3 | 1 | 0.2 | 10 | 12 | 9 |
| 4 | 0 | 0.8 | 14 | 19 | 15 |
| 5 | 0 | 0.5 | 12 | 16 | 12 |
| 6 | 0 | 0.2 | 8 | 12 | 9 |

*Your work:*

### Exercise 11 · Regression and AIPW estimates

Compute the regression estimate and the AIPW estimate of the ATE.

<details><summary>Show solution</summary>

**Step 1.** Regression: $\hat\mu_1 - \hat\mu_0$ = 4, 4, 3, 4, 4, 3; mean $22/6 \approx 3.67$.

**Step 2.** AIPW terms, treated ($+\,(Y - \hat\mu_1)/\hat e$): household 1: $4 + 1/0.8 = 5.25$; household 2: $4 + 0 = 4$; household 3: $3 + (-2)/0.2 = -7$.

**Step 3.** Controls ($-\,(Y - \hat\mu_0)/(1 - \hat e)$): household 4: $4 - (-1)/0.2 = 9$; household 5: $4 - 0 = 4$; household 6: $3 - (-1)/0.8 = 4.25$.

**Step 4.** Mean: $(5.25 + 4 - 7 + 9 + 4 + 4.25)/6 = 19.5/6 = 3.25$.

**Answer:** regression 3.67, AIPW 3.25 (Day 44: naive 4.0, Hájek 0.36). With six households the estimators disagree a lot; their disagreement is itself a warning about model dependence and overlap.

</details>

*Your work:*

### Exercise 12 · A wrong outcome model, a right propensity model

In simulation (code below), the true earnings model is non-linear in age, the analyst fits a linear outcome model, and the propensity model is correct. Which estimators are biased?

<details><summary>Show solution</summary>

**Step 1.** Regression relies on the (wrong) linear outcome model: biased.

**Step 2.** IPW relies on the (correct) propensity model: consistent.

**Step 3.** AIPW is consistent through the propensity model (Exercise 7).

**Step 4.** The simulation confirms it: regression averages about 1.44 against a truth of 2, while IPW and AIPW average about 1.98.

**Step 5.** Precision: here AIPW's SD (≈ 0.43) is slightly *larger* than IPW's (≈ 0.38). The linear model misses the U-shape in age, so its residuals are large, and weighting large residuals adds noise.

**Answer:** double robustness delivered consistency; the precision gain of AIPW needs a reasonably good outcome model. Flexible outcome models (splines, machine learning) fix this, which is the motivation for Day 90.

</details>

*Your work:*

### Exercise 13 · Both models wrong

What happens if both models are misspecified, and how can analysts reduce that risk?

<details><summary>Show solution</summary>

**Step 1.** No guarantee: the bias is the product of the two errors (Exercise 10), which can still be substantial.

**Step 2.** Remedies: flexible models (splines, interactions, machine learning) for both nuisances, with cross-fitting; balance diagnostics for the propensity model; and checking that regression, IPW and AIPW roughly agree.

**Answer:** double robustness gives two chances, not immunity.

</details>

*Your work:*

### Exercise 14 · Writing the policy report

A ministry asks whether a youth employment subsidy raised employment. You have administrative data on all eligible youths, rich pre-programme records and the eligibility rules. Outline the Phase 3 analysis plan.

<details><summary>Show solution</summary>

**Step 1.** Estimand: ATT (is the subsidy worth it for those who used it?) and, for an expansion, the ATE or CATEs (Day 33).

**Step 2.** Identification: argue unconfoundedness from the eligibility rules and include all variables officials used, plus lagged employment (Day 42).

**Step 3.** Overlap: estimate propensities, inspect histograms and normalised differences, and trim if needed (Days 42–43).

**Step 4.** Estimation: AIPW with flexible models and cross-fitting; report regression and IPW as checks (Days 44–45).

**Step 5.** Credibility: placebo test on pre-programme employment, sensitivity analysis for unobserved confounding.

**Answer:** a transparent chain from estimand to assumptions to diagnostics to estimator, with robustness checks.

</details>

*Your work:*

### Exercise 15 · Match the tool to the problem

For each case, name the Phase 3 tool: (a) 12 clinics, 6 randomised to a new protocol, outcome is patient satisfaction; (b) a lottery for vouchers where 30% of winners do not use them; (c) a trial randomising households within villages where neighbours share information; (d) an observational study with 50 baseline covariates and good overlap; (e) an RCT with a strong baseline measure of the outcome.

<details><summary>Show solution</summary>

**Step 1.** (a) Cluster randomisation with clinic-level randomisation inference (Days 37, 40).

**Step 2.** (b) ITT for the voucher offer and CACE = ITT_Y / ITT_D for users (Day 41).

**Step 3.** (c) Spillovers: estimate direct and spillover effects, or randomise villages (Days 32, 40).

**Step 4.** (d) Propensity score with AIPW and balance checks (Days 43–45).

**Step 5.** (e) Lin's covariate adjustment, or block on the baseline measure (Days 38–39).

</details>

## Verify in Python

*Your work:*

In [ ]:
import numpy as np
from scipy import optimize

T = np.array([1, 1, 1, 0, 0, 0]); e = np.array([.8, .5, .2, .8, .5, .2]); Y = np.array([20, 16, 10, 14, 12, 8.])
m1 = np.array([19, 16, 12, 19, 16, 12.]); m0 = np.array([15, 12, 9, 15, 12, 9.])
psi = m1 - m0 + T * (Y - m1) / e - (1 - T) * (Y - m0) / (1 - e)
print("psi:", psi, " regression:", round((m1 - m0).mean(), 3), " AIPW:", round(psi.mean(), 3))

# Monte Carlo: wrong outcome model, correct propensity model
rng = np.random.default_rng(45)
out = {"regression": [], "IPW (Hajek)": [], "AIPW": []}
for _ in range(1500):
    n = 1000; age = rng.uniform(18, 60, n); X = np.column_stack([np.ones(n), age])
    e_true = 1 / (1 + np.exp(-(3 - 0.1 * age))); T = rng.random(n) < e_true
    y = 10 + 0.02 * (age - 35)**2 + 2 * T + rng.normal(0, 2, n)          # non-linear in age; effect 2
    b = optimize.minimize(lambda b: -np.sum(T * (X @ b) - np.log1p(np.exp(X @ b))), np.zeros(2)).x
    eh = 1 / (1 + np.exp(-X @ b))
    b1 = np.linalg.lstsq(X[T], y[T], rcond=None)[0]; b0 = np.linalg.lstsq(X[~T], y[~T], rcond=None)[0]
    mu1, mu0 = X @ b1, X @ b0                                              # linear (wrong) outcome models
    out["regression"].append((mu1 - mu0).mean())
    out["IPW (Hajek)"].append((T * y / eh).sum() / (T / eh).sum() - ((~T) * y / (1 - eh)).sum() / ((~T) / (1 - eh)).sum())
    out["AIPW"].append(np.mean(mu1 - mu0 + T * (y - mu1) / eh - (~T) * (y - mu0) / (1 - eh)))
for k, v in out.items():
    print(f"{k:12s} mean {np.mean(v):.3f} (truth 2)  SD {np.std(v):.3f}")

## Key takeaways

- AIPW adds an inverse-probability-weighted correction to an outcome-model estimate.
- It is consistent if either model is right, and efficient if both are; its bias is a product of the two models' errors.
- Phase 3 gives you the design-based and model-based toolkit; Phase 4 adds graphs to decide what to condition on.